# Decisions and Uncertainty

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

## Comparing Distributions: Be the Jury Program

In [ ]:
# Counts are from the 88% of participants in Be the Jury in 2023 who completed the exit survey and shared their race

sf = Table().with_columns(
    'Race/Ethnicity',
    ['American Indian or Alaska Native',
     'Asian',
     'Black or African American',
     'Hispanic or Latino',
     'Native Hawaiian or Other Pacific Islander',
     'Some Other Race',
     'White'],
    'Count',[4, 168,23,59,4,5,175],
    'SF Census (2020)', [0.005, 0.3440, 0.0520, 0.1540, 0.0040, 0.0490, 0.3920])
sf.set_format(2, PercentFormatter)

In [ ]:
sum(sf.column('Count'))

In [ ]:
sf = (sf.with_column('Panels', sf.column('Count') / sum(sf.column('Count')))
        .relabeled('SF Census (2020)', 'Eligible'))
sf.set_format(3, PercentFormatter)

In [ ]:
sf.select(0, 2, 3).barh(0)

In [ ]:
def tvd(dist1, dist2):
    return sum(abs(dist1 - dist2))/2

jury = sf
model = jury.column('Eligible')
obsvd_tvd = tvd(jury.column('Panels'), model)
obsvd_tvd

In [ ]:
model = jury.column('Eligible')

def simulated_tvd():
    return tvd(sample_proportions(438, model), model)

tvds = make_array()

num_simulations = 10000
for i in np.arange(num_simulations):
    new_tvd = simulated_tvd()
    tvds = np.append(tvds, new_tvd)

In [ ]:
title = 'Simulated TVDs (if model is true)'
bins = np.arange(0, .1, .005)

Table().with_column(title, tvds).hist(bins = bins)
print('Observed TVD: ' + str(obsvd_tvd))

# Plotting details; ignore this code
plots.ylim(-2, 55)
plots.scatter(obsvd_tvd, 0, color='red', s=30);

**DISCUSS**: Comparing distributions
1. What are the two viewpoints compared above?
    * Null hypothesis: _______.
    * Alternative hypothesis: ________.
1. Why did we choose total variation distance (TVD) as our statistic?
1. Is the observed TVD consistent with the model presented in the null hypothesis?
1. Bonus: Why does TVD have a factor of 2?

## The GSI's Defense ##

In [ ]:
scores = Table.read_table('scores_by_section.csv')
scores.show(10)

In [ ]:
max(scores.column('Midterm'))

In [ ]:
min(scores.column('Midterm'))

In [ ]:
scores.group('Section')

In [ ]:
scores.group('Section', np.average).show()

## Define the null hypothesis and alternative hypothesis 

See slides

## Choose a test statistic

**DISCUSS**: What test statistic should we use?

<br/><br/><br/>

In [ ]:
# test statistic: ___________
observed_average = scores.group('Section', np.average).where('Section', 3).column(1).item(0)
observed_average

Conducting a hypothesis test will help us __________

## Find the empirical distribution of the statistic under the null hypothesis

The two viewpoints, restated slightly:

* Null hypothesis: The midterm scores for section 3 are like a random draw of _______ students from the course
* Alternative hypothesis: The midterm scores for section 3 are too low to be explained well by randomness alone.

In [ ]:
# (1) Simulate what section 3 could have looked like under the null hypothesis,
# i.e., under the hypothesis that the section is like a random sample from the class

random_sample = scores.sample(..., with_replacement=False)
random_sample

In [ ]:
# (2) Compute the *test statistic* for this simulated sample
np.average(random_sample.column('Midterm'))

In [ ]:
# (3) Write a function that combines steps (1) and (2)

def random_sample_midterm_avg():
    """
    Simulates one value of the test statistic 
    under the hypothesis that the section is like a random sample from the class
    """
    random_sample = scores.sample(27, with_replacement = False)
    return np.average(random_sample.column('Midterm'))

In [ ]:
# (4) Simulate 100,000 copies of the test statistic

sample_averages = make_array()

for i in np.arange(100000):
    sample_averages = np.append(sample_averages, random_sample_midterm_avg())    

**DISCUSS**:
1. Why did we sample **without** replacement?
1. Why did we simulate 100,000 times? Why not 27 times?

## Our Decision

In [ ]:
# Compare the simulated distribution of the statistic
# and the actual observed statistic
averages_tbl = Table().with_column('Random Sample Average', sample_averages)
averages_tbl.hist(bins = 20)
plots.scatter(observed_average, -0.01, color='red', s=120);

### Approach 1: Compute the p-value

In [ ]:
# (1) Calculate the p-value: simulation area beyond observed value
np.count_nonzero(sample_averages <= observed_average) / len(sample_averages)
# (2) See if this is less than the threshold, 5%

### Approach 2: Compute the threshold

In [ ]:
# (1) Find simulated value corresponding to 5% of simulated values
five_percent_point = averages_tbl.sort(0).column(0).item(round(len(sample_averages) * 0.05))
five_percent_point

In [ ]:
# (2) See if this value is greater than observed value
observed_average

### Visual Representation

In [ ]:
averages_tbl.hist(bins = 20)
plots.plot([five_percent_point, five_percent_point], [0, 0.35], color='gold', lw=2)
plots.title('Area to the left of the gold line: 5%');
plots.scatter(observed_average, -0.01, color='red', s=120);